<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/LegalRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Install Google GenAI SDK and supporting libraries while respecting Colab's pinned google-auth
!pip install -q "google-auth==2.49.0" google-genai faiss-cpu sentence-transformers langchain-text-splitters

In [6]:
import os
import numpy as np
import faiss
from google import genai
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata

# Initialize Gemini Client
# Make sure to store your GEMINI_API_KEY in Colab's "Secrets" tab (key icon on the left panel)
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("Gemini Client successfully initialized!")
except Exception as e:
    print(f"Error loading API Key. Please make sure GEMINI_API_KEY is saved in Colab Secrets: {e}")

Gemini Client successfully initialized!


In [7]:
# Sample Legal Case Law Documents
legal_documents = [
    {
        "id": "doc1",
        "title": "State v. Johnson",
        "metadata": {"case_name": "State v. Johnson", "year": 2021, "jurisdiction": "State Supreme Court"},
        "text": """
        The defendant was charged with grand larceny under Statute § 402. The defense argued that consent was implied based on prior commercial dealings between the parties. However, the prosecution demonstrated that consent was explicitly revoked in writing on March 12, 2020. The Court held that implied consent cannot override an explicit, written revocation of consent. Furthermore, property values exceeded the statutory threshold of $5,000 for grand larceny, establishing intent and fulfilling all statutory elements for a felony conviction.
        """
    },
    {
        "id": "doc2",
        "title": "Apex Tech Enterprises v. DataCorp",
        "metadata": {"case_name": "Apex Tech Enterprises v. DataCorp", "year": 2023, "jurisdiction": "Federal District Court"},
        "text": """
        In this trade secret dispute, Apex Tech claimed that DataCorp misappropriated proprietary source code. DataCorp filed a motion to dismiss, citing failure to state a claim under Rule 12(b)(6). The Court analyzed whether the source code was subject to reasonable secrecy measures under the Defend Trade Secrets Act (DTSA). The court found that non-disclosure agreements (NDAs) and role-based access control systems implemented by Apex Tech constituted reasonable security measures. Consequently, DataCorp's motion to dismiss was denied, and the case proceeded to discovery.
        """
    },
    {
        "id": "doc3",
        "title": "Smith v. City Transit Authority",
        "metadata": {"case_name": "Smith v. City Transit Authority", "year": 2019, "jurisdiction": "Appellate Court"},
        "text": """
        The plaintiff sued for personal injury sustained after a slip and fall at a subway station. The defendant invoked sovereign immunity. The Appellate Court reviewed whether municipal negligence in public infrastructure maintenance constitutes an exception to government tort liability. The court held that failure to maintain public safety facilities in reasonable condition falls under the statutory exception to sovereign immunity under Civil Rights Law § 1983. Summary judgment in favor of the City Transit Authority was reversed, and the matter was remanded for trial.
        """
    }
]

# Initialize text splitter (chunk size of 200 characters with 30-character overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len
)

# Split documents into chunks and bind metadata to each chunk
chunked_docs = []
for doc in legal_documents:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "text": chunk.strip(),
            "metadata": doc["metadata"]
        })

print(f"Total documents processed: {len(legal_documents)}")
print(f"Total chunks created: {len(chunked_docs)}")
print("\nSample Chunk structure:")
print(chunked_docs[0])

Total documents processed: 3
Total chunks created: 11

Sample Chunk structure:
{'chunk_id': 'doc1_chunk_0', 'text': 'The defendant was charged with grand larceny under Statute § 402. The defense argued that consent was implied based on prior commercial dealings between the parties. However, the prosecution', 'metadata': {'case_name': 'State v. Johnson', 'year': 2021, 'jurisdiction': 'State Supreme Court'}}


In [11]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- SELF-HEALING / STATE RECOVERY CHECK ---
# If runtime restarted, recreate chunked_docs, embedding_model, and faiss_index automatically
if 'chunked_docs' not in globals() or 'faiss_index' not in globals() or 'embedding_model' not in globals():
    print("Initializing RAG models and index in memory...")

    # 1. Define legal documents
    legal_documents = [
        {
            "id": "doc1",
            "title": "State v. Johnson",
            "metadata": {"case_name": "State v. Johnson", "year": 2021, "jurisdiction": "State Supreme Court"},
            "text": "The defendant was charged with grand larceny under Statute § 402. The defense argued that consent was implied based on prior commercial dealings between the parties. However, the prosecution demonstrated that consent was explicitly revoked in writing on March 12, 2020. The Court held that implied consent cannot override an explicit, written revocation of consent. Furthermore, property values exceeded the statutory threshold of $5,000 for grand larceny, establishing intent and fulfilling all statutory elements for a felony conviction."
        },
        {
            "id": "doc2",
            "title": "Apex Tech Enterprises v. DataCorp",
            "metadata": {"case_name": "Apex Tech Enterprises v. DataCorp", "year": 2023, "jurisdiction": "Federal District Court"},
            "text": "In this trade secret dispute, Apex Tech claimed that DataCorp misappropriated proprietary source code. DataCorp filed a motion to dismiss, citing failure to state a claim under Rule 12(b)(6). The Court analyzed whether the source code was subject to reasonable secrecy measures under the Defend Trade Secrets Act (DTSA). The court found that non-disclosure agreements (NDAs) and role-based access control systems implemented by Apex Tech constituted reasonable security measures. Consequently, DataCorp's motion to dismiss was denied, and the case proceeded to discovery."
        },
        {
            "id": "doc3",
            "title": "Smith v. City Transit Authority",
            "metadata": {"case_name": "Smith v. City Transit Authority", "year": 2019, "jurisdiction": "Appellate Court"},
            "text": "The plaintiff sued for personal injury sustained after a slip and fall at a subway station. The defendant invoked sovereign immunity. The Appellate Court reviewed whether municipal negligence in public infrastructure maintenance constitutes an exception to government tort liability. The court held that failure to maintain public safety facilities in reasonable condition falls under the statutory exception to sovereign immunity under Civil Rights Law § 1983. Summary judgment in favor of the City Transit Authority was reversed, and the matter was remanded for trial."
        }
    ]

    # 2. Chunk documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30, length_function=len)
    chunked_docs = []
    for doc in legal_documents:
        chunks = text_splitter.split_text(doc["text"])
        for i, chunk in enumerate(chunks):
            chunked_docs.append({
                "chunk_id": f"{doc['id']}_chunk_{i}",
                "text": chunk.strip(),
                "metadata": doc["metadata"]
            })

    # 3. Load embedding model and create FAISS index
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    chunk_texts = [c["text"] for c in chunked_docs]
    embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True).astype('float32')

    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(embeddings)
    print("Setup complete! Memory restored.\n")


# --- RETRIEVAL AND RANKING FUNCTION ---
def retrieve_and_rank_chunks(query, k=3):
    """
    Retrieves top k chunks for a given query, calculates similarity scores,
    and formats them with their associated metadata.
    """
    query_vector = embedding_model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = faiss_index.search(query_vector, k)

    retrieved_results = []
    for rank, idx in enumerate(indices[0]):
        chunk = chunked_docs[idx]
        distance = distances[0][rank]
        similarity_score = 1 / (1 + distance)

        retrieved_results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "metadata": chunk["metadata"],
            "l2_distance": float(distance),
            "similarity_score": float(similarity_score)
        })

    return retrieved_results

# --- RUN RETRIEVAL TEST ---
sample_query = "What constitutes reasonable security measures for trade secrets?"
retrieved_chunks = retrieve_and_rank_chunks(sample_query, k=3)

print(f"Query: '{sample_query}'\n")
print("Top Retrieved & Ranked Chunks:")
for item in retrieved_chunks:
    print(f"Rank {item['rank']} | Score: {item['similarity_score']:.4f} | Case: {item['metadata']['case_name']} ({item['metadata']['year']})")
    print(f"Text: {item['text']}\n")

Initializing RAG models and index in memory...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete! Memory restored.

Query: 'What constitutes reasonable security measures for trade secrets?'

Top Retrieved & Ranked Chunks:
Rank 1 | Score: 0.5437 | Case: Apex Tech Enterprises v. DataCorp (2023)
Text: under Rule 12(b)(6). The Court analyzed whether the source code was subject to reasonable secrecy measures under the Defend Trade Secrets Act (DTSA). The court found that non-disclosure agreements

Rank 2 | Score: 0.4233 | Case: Apex Tech Enterprises v. DataCorp (2023)
Text: non-disclosure agreements (NDAs) and role-based access control systems implemented by Apex Tech constituted reasonable security measures. Consequently, DataCorp's motion to dismiss was denied, and

Rank 3 | Score: 0.3940 | Case: Apex Tech Enterprises v. DataCorp (2023)
Text: In this trade secret dispute, Apex Tech claimed that DataCorp misappropriated proprietary source code. DataCorp filed a motion to dismiss, citing failure to state a claim under Rule 12(b)(6). The



In [12]:
def generate_legal_answer(query, k=3):
    """
    RAG Pipeline: Retrieves relevant chunks, builds a structured prompt,
    and calls Gemini 2.5 Flash to generate a grounded legal answer.
    """
    # 1. Retrieve top matching chunks
    retrieved_chunks = retrieve_and_rank_chunks(query, k=k)

    # 2. Construct context string with metadata
    context_blocks = []
    for chunk in retrieved_chunks:
        meta = chunk["metadata"]
        block = (
            f"[Source: {meta['case_name']} ({meta['year']}), Jurisdiction: {meta['jurisdiction']}]\n"
            f"Content: {chunk['text']}"
        )
        context_blocks.append(block)

    context_str = "\n\n".join(context_blocks)

    # 3. Create RAG prompt for Gemini
    prompt = f""" You are an expert legal assistant. Answer the user's query based strictly on the provided retrieved case law context.
If the context does not contain enough information to answer the question, state clearly that the answer is not present in the provided documents.

RETRIEVED CASE LAW CONTEXT:
{context_str}

USER QUERY:
{query}

INSTRUCTIONS:
- Provide a clear, concise, and direct legal summary.
- Explicitly cite the case name, year, and jurisdiction when referencing facts or holdings.
"""

    # 4. Generate response using Gemini 2.5 Flash
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text, retrieved_chunks

# Test the RAG pipeline with sample queries
test_queries = [
    "What constitutes reasonable security measures for trade secrets?",
    "Under what conditions is sovereign immunity waived for municipal transit authorities?",
    "Can implied consent override an explicit written revocation in larceny cases?"
]

for q in test_queries:
    print("=" * 80)
    print(f"QUERY: {q}\n")
    answer, chunks = generate_legal_answer(q, k=2)
    print("GENERATED ANSWER:")
    print(answer)
    print("\n" + "=" * 80 + "\n")

QUERY: What constitutes reasonable security measures for trade secrets?

GENERATED ANSWER:
According to *Apex Tech Enterprises v. DataCorp* (2023), a Federal District Court case, non-disclosure agreements (NDAs) and role-based access control systems constituted reasonable security measures for trade secrets.


QUERY: Under what conditions is sovereign immunity waived for municipal transit authorities?

GENERATED ANSWER:
According to *Smith v. City Transit Authority (2019)*, an Appellate Court case, sovereign immunity for municipal transit authorities is waived when municipal negligence in maintaining facilities in reasonable condition falls under the statutory exception to sovereign immunity under Civil Rights Law § 1983.


QUERY: Can implied consent override an explicit written revocation in larceny cases?

GENERATED ANSWER:
No, implied consent cannot override an explicit, written revocation of consent. The State Supreme Court held in *State v. Johnson (2021)* that implied consent can